# Baselines - Node Classification - `dgl` lane

One lane of the baseline campaign: every node classification model that needs
a self-contained PyTorch 2.4/CUDA 12.4 + DGL installation cell.

- **hinormer** on acm -- The authors' own repository, not H2GB's copy, whose transformer stack is commented out.

**1 cells.** Lanes are independent, so this notebook installs only its own
group and can run on its own Colab runtime alongside the others. To split this lane
further across machines, keep `NUM_RUNS` and `BASE_SEED` fixed and give each machine a
disjoint `UNIT_START`/`UNIT_COUNT` range; never run the same unit index twice.

Generated by `scripts/generate_baseline_notebooks.py` - edit that, not this file.

## 0. Environment

In [1]:
from pathlib import Path
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/H2S-NN_test')
else:
    PROJECT_DIR = Path.cwd()

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('Working directory:', PROJECT_DIR)

Mounted at /content/drive
Working directory: /content/drive/MyDrive/H2S-NN_test


In [7]:
pip install  dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html

Looking in links: https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html
  Using cached dgl-0.1.3-py3-none-manylinux1_x86_64.whl.metadata (497 bytes)
Using cached dgl-0.1.3-py3-none-manylinux1_x86_64.whl (409 kB)


In [9]:
import torch
print(torch.version.cuda)


12.8


In [11]:
import dgl
import torch

print("DGL version:", dgl.__version__)
try:
    g = dgl.graph(([0], [1]))
    g = g.to('cuda')
    print("GPU is enabled for DGL.")
except Exception as e:
    print("GPU is NOT enabled. Error:", e)

ImportError: cannot import name 'Mapping' from 'collections' (/usr/lib/python3.13/collections/__init__.py)

This lane installs its own PyTorch 2.4/CUDA 12.4, PyG, and DGL dependencies directly
in the next cell. It does not use `requirements.txt` or `requirements/core.txt`: those
pin torch 2.6, which is incompatible with the required DGL wheel repository. A fresh
Colab runtime restarts after torch is replaced; **rerun from the first cell once it
restarts**.

In [2]:
# This lane is intentionally self-contained. DGL's CUDA 12.4 wheel must match torch
# 2.4, so do not install requirements/core.txt (which pins this project to torch 2.6).
# The DGL command below deliberately uses DGL's torch-2.4/cu124 wheel repository.
import importlib.metadata
import os
import signal
import subprocess
import sys

TORCH_INDEX = 'https://download.pytorch.org/whl/cu124'
PYG_WHEEL_INDEX = 'https://data.pyg.org/whl/torch-2.4.0+cu124.html'
DGL_WHEEL_INDEX = 'https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html'

try:
    installed_torch = importlib.metadata.version('torch')
except importlib.metadata.PackageNotFoundError:
    installed_torch = ''
target_ready = installed_torch.startswith('2.4.') and '+cu124' in installed_torch
print('installed torch distribution:', installed_torch or 'not installed')

commands = [
    [sys.executable, '-m', 'pip', 'install', '--upgrade',
     'tqdm', 'numpy', 'scipy', 'pyyaml', 'scikit-learn', 'wandb', 'pandas',
     'umap-learn', 'matplotlib'],
    [sys.executable, '-m', 'pip', 'install', '--upgrade', '--force-reinstall',
     '--no-cache-dir', '--index-url', TORCH_INDEX, 'torch==2.4.*'],
    [sys.executable, '-m', 'pip', 'install', '--upgrade', '--force-reinstall',
     '--no-cache-dir', '-f', PYG_WHEEL_INDEX,
     'torch_geometric', 'torch_scatter', 'torch_sparse'],
    # Required exact DGL wheel source for torch 2.4 + CUDA 12.4.
    [sys.executable, '-m', 'pip', 'install', '--upgrade', '--force-reinstall',
     '--no-cache-dir', 'dgl', '-f', DGL_WHEEL_INDEX],
]

for command in commands:
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-3000:])
        print(result.stderr[-3000:])
        raise RuntimeError(f'Install failed with exit code {result.returncode}: {" ".join(command)}')
print('PyTorch 2.4/CUDA 12.4, PyG, DGL, and notebook dependencies installed.')

if not target_ready:
    print('PyTorch was replaced. Restarting; rerun this notebook from the first cell.')
    if IN_COLAB:
        os.kill(os.getpid(), signal.SIGKILL)
    raise RuntimeError('Restart Python, then rerun the notebook from the first cell.')

installed torch distribution: 2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu124

ERROR: Could not find a version that satisfies the requirement torch==2.4.* (from versions: 2.5.0+cu124, 2.5.1+cu124, 2.6.0+cu124)
ERROR: No matching distribution found for torch==2.4.*



RuntimeError: Install failed with exit code 1: /usr/bin/python3 -m pip install --upgrade --force-reinstall --no-cache-dir --index-url https://download.pytorch.org/whl/cu124 torch==2.4.*

In [ ]:
# Confirm the compiled extensions after the restart requested by the install cell.
import torch
import torch_geometric
import torch_scatter  # noqa: F401  (proves the compiled torch ABI matches)
import torch_sparse  # noqa: F401

if not torch.__version__.startswith('2.4.'):
    raise RuntimeError(
        f'Expected torch 2.4.*, found {torch.__version__}. Rerun the install cell.'
    )
if torch.version.cuda != '12.4':
    raise RuntimeError(f'Expected the CUDA 12.4 build, found CUDA {torch.version.cuda}.')

from src.baselines.runtime.dependencies import REQUIREMENT_GROUPS

GROUP = REQUIREMENT_GROUPS['dgl']
GROUP_OK = GROUP.is_installed()
if not GROUP_OK:
    raise RuntimeError('DGL did not install from the torch-2.4/cu124 wheel repository.')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch:', torch.__version__, '| pyg:', torch_geometric.__version__, '| device:', DEVICE)
if DEVICE == 'cuda':
    properties = torch.cuda.get_device_properties(0)
    print('GPU:', properties.name, '|', round(properties.total_memory / 1024**3, 1), 'GiB')
else:
    print('\nWARNING: no GPU on this runtime. Runtime > Change runtime type > GPU.')

In [ ]:
# DGL was installed directly in the self-contained environment cell above.
# Its availability and CUDA ABI are checked in the following cell.

In [ ]:
# DGL ships separate CPU and CUDA builds and `import dgl` cannot tell them apart. When
# the CPU build is the one present, `resolve_dgl_device` returns CPU without raising and
# every model in this lane trains on the CPU of a GPU machine -- correct numbers, wrong
# hardware, no error. On Linux the +cu124 wheel exists, so that is a broken install and
# this cell refuses to continue rather than burn a session discovering it in the timings.
import sys

from src.baselines.runtime.dependencies import dgl_cuda_report

if not GROUP_OK:
    raise RuntimeError(
        f'{GROUP.name} did not install, and every model in this lane needs it.'
    )

uses_gpu, explanation = dgl_cuda_report()
print(('OK   ' if uses_gpu else 'WARN '), explanation)
if not uses_gpu and DEVICE == 'cuda' and sys.platform == 'linux':
    raise RuntimeError(
        'Refusing to run the DGL lane on CPU while a GPU is present. '
        + explanation
    )
print('Lane ready.')

In [ ]:
# The vendored baselines are cloned at the repository root, not pip-installed. Check the
# ones this lane reads before anything trains.
from src.baselines.runtime import available_vendors

REQUIRED_VENDORS = ['hinormer']
present = available_vendors()
for name in REQUIRED_VENDORS:
    print(('OK      ' if present.get(name) else 'MISSING '), name)
missing = [name for name in REQUIRED_VENDORS if not present.get(name)]
if missing:
    raise RuntimeError(f'Clone these at the repository root first: {missing}')

In [ ]:
import wandb

wandb_key = None
if IN_COLAB:
    try:
        from google.colab import userdata
        wandb_key = userdata.get('WANDB_API_KEY')
    except Exception:
        wandb_key = None

if wandb_key:
    wandb.login(key=wandb_key, relogin=True)
else:
    wandb.login()

## 1. Campaign settings

In [ ]:
TASK = 'node_classification'
LANE = 'dgl'

# (model, datasets) for this lane only.
CELLS = [
    ('hinormer', ['acm']),
]

N_TRIALS = 30
NUM_RUNS = 5
BASE_SEED = 42

# Units are seeds. Give each machine a disjoint range; never run one index twice.
UNIT_START = 0
UNIT_COUNT = None

RESUME = False
OUTPUT_DIR = Path('experiments/baselines')
WANDB_PROJECT = 'h2snn_baseline'
WANDB_MODE = 'online'
WANDB_TAGS = ['baselines', TASK, f'lane-{LANE}']

# Epochs and patience are protocol, not settings: they come from
# src/baselines/budget.py so every model in a table trains under the same budget.
# Passing --epochs here would silently override that for this lane only.
from src.baselines.budget import default_budget

budget = default_budget(TASK)
print(f'{sum(len(d) for _, d in CELLS)} cells x {NUM_RUNS} seeds x {N_TRIALS} trials')
print(f'budget: {budget.epochs} epochs, patience {budget.patience}')

Drop any model whose requirement group did not install. This happens **before** the
launch cells, so a missing library costs those cells and nothing else, and it is
reported rather than silent.

In [ ]:
from src.baselines.runtime.dependencies import models_missing_dependencies

requested = [model for model, _ in CELLS]
blocked = models_missing_dependencies(tuple(requested), TASK)
CELLS = [(model, datasets) for model, datasets in CELLS if model not in blocked]

for name, reason in blocked.items():
    print(f'SKIP  {name}: {reason}')
print(f'{len(CELLS)} of {len(requested)} runnable: ' + ', '.join(m for m, _ in CELLS))
if not CELLS:
    raise RuntimeError('No runnable models in this lane; fix the install failure above.')

## 2. Scope

Every cell must be runnable, and each is labelled with whether its authors published
settings for that exact dataset. Published cells go through `run_baseline_published.py`
in section 3; the rest are tuned in section 4.

In [ ]:
from src.baselines.registry import get_adapter, unsupported_reason

PUBLISHED, TUNED = [], []
for model, datasets in CELLS:
    adapter = get_adapter(model)
    for dataset in datasets:
        reason = unsupported_reason(model, TASK, dataset)
        assert reason is None, f'{model}/{dataset} is not runnable: {reason}'
        published = adapter.published_params(TASK, dataset)
        space = adapter.search_space(TASK, dataset)
        (PUBLISHED if published else TUNED).append((model, dataset))
        status = 'published' if published else 'tune'
        print(f'{model:<12}{dataset:<10}{len(space):>3} params  {status}')
print(f'\n{len(PUBLISHED)} published, {len(TUNED)} to tune')

## 3. Published-hyperparameter runs

Cells whose authors published settings for this exact dataset. No sweep: five seeds at
their values, using **their** epoch budget rather than the campaign's, because
reproducing a reported number means reproducing the budget it was reported at. Writes
`published_summary.json`, so it never overwrites a tuned result.

In [ ]:
import subprocess
import sys

if PUBLISHED:
    by_model = {}
    for model, dataset in PUBLISHED:
        by_model.setdefault(model, []).append(dataset)
    for model, datasets in by_model.items():
        cmd = [
            sys.executable, 'scripts/run_baseline_published.py',
            '--task', TASK,
            '--models', model,
            '--datasets', *datasets,
            '--output_dir', str(OUTPUT_DIR),
            '--num_runs', str(NUM_RUNS),
            '--base_seed', str(BASE_SEED),
        ]
        print(' '.join(cmd))
        subprocess.run(cmd, check=True)
else:
    print('No published cells in this lane.')

## 4. Tuned runs

`--validate_only` first: it resolves every search space and prints the fit count without
running anything.

In [ ]:
def tuning_command(model, datasets, validate=False):
    cmd = [
        sys.executable, 'scripts/run_baseline_tuning.py',
        '--task', TASK,
        '--models', model,
        '--datasets', *datasets,
        '--output_dir', str(OUTPUT_DIR),
        '--n_trials', str(N_TRIALS),
        '--num_runs', str(NUM_RUNS),
        '--base_seed', str(BASE_SEED),
        '--unit_start', str(UNIT_START),
        '--wandb_project', WANDB_PROJECT,
        '--wandb_mode', WANDB_MODE,
        '--wandb_tags', *WANDB_TAGS,
    ]
    if UNIT_COUNT is not None:
        cmd += ['--unit_count', str(UNIT_COUNT)]
    if RESUME:
        cmd.append('--resume')
    if validate:
        cmd.append('--validate_only')
    return cmd


TUNE_BY_MODEL = {}
for model, dataset in TUNED:
    TUNE_BY_MODEL.setdefault(model, []).append(dataset)

for model, datasets in TUNE_BY_MODEL.items():
    subprocess.run(tuning_command(model, datasets, validate=True), check=True)

One launch cell per model, so a failure costs one model rather than the lane. After a
Colab restart set `RESUME = True` in section 1 and re-run the cell: compatible per-seed
states resume their original sweeps, and incompatible ones are preserved while numbered
continuation sweeps are created.

### hinormer - ACM

The authors' own repository, not H2GB's copy, whose transformer stack is commented out.

In [ ]:
model, datasets = 'hinormer', TUNE_BY_MODEL.get('hinormer', [])
if datasets:
    cmd = tuning_command(model, datasets)
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('hinormer: nothing to tune in this lane (published cells ran in section 3).')

## 5. Aggregate

Collects whatever has finished into one CSV for this lane. Cells that have not run are
absent rather than filled in, so a gap stays visibly a gap.

In [ ]:
import json

import pandas as pd

from src.utils.run_reports import report_metrics_for_task

metrics = report_metrics_for_task(TASK)
wanted = {f'{m}_{TASK}_{d}' for m, ds in CELLS for d in ds}

records = []
for pattern in ('*/tuning_summary.json', '*/published_summary.json'):
    for summary_path in sorted(OUTPUT_DIR.glob(pattern)):
        if summary_path.parent.name not in wanted:
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))['summary']
        model, _, dataset = summary_path.parent.name.partition(f'_{TASK}_')
        row = {
            'model': model,
            'dataset': dataset,
            'source': 'published' if 'published' in summary_path.name else 'tuned',
            'seeds': summary.get('num_units', 0),
        }
        for metric in metrics:
            entry = summary.get('metrics', {}).get(metric, {})
            row[metric] = entry.get('mean')
            row[f'{metric}_std'] = entry.get('std')
        records.append(row)

frame = pd.DataFrame(records)
results_path = Path('results') / f'baselines_{TASK}_{LANE}.csv'
results_path.parent.mkdir(parents=True, exist_ok=True)
frame.to_csv(results_path, index=False)
print(f'{len(frame)} of {len(wanted)} cells finished -> {results_path}')
frame

In [ ]:
# Manuscript form: mean +/- std on the headline metric this task reports.
if not frame.empty:
    headline = metrics[0]
    shown = frame.assign(
        value=frame.apply(
            lambda r: f"{100 * r[headline]:.2f} +/- {100 * r[f'{headline}_std']:.2f}",
            axis=1,
        )
    )
    print(shown.pivot_table(index='model', columns='dataset', values='value',
                            aggfunc='first').to_string())
else:
    print('Nothing finished yet - run section 3 and 4 first.')

## 6. Resume and partition

Per-seed state lives under `OUTPUT_DIR/tuning_state/<cell>/`. Set `RESUME = True` in
section 1 and re-run the relevant launch cell to continue. `pruned` counts trials
Hyperband stopped early, which is a normal outcome and not a failure.

In [ ]:
state_root = OUTPUT_DIR / 'tuning_state'
for state_path in sorted(state_root.glob('*/tuning_state_*.json')):
    if state_path.parent.name not in wanted or 'worker' in state_path.stem:
        continue
    state = json.loads(state_path.read_text(encoding='utf-8'))
    print(
        f"{state_path.parent.name:<40}{state_path.stem:<26}"
        f"done {len(state.get('records', []))}"
        f" attempted {state.get('attempted_trials', 0)}"
        f" pruned {state.get('pruned_trials', 0)}"
        f" failed {state.get('failed_trials', 0)}"
        f"  sweep {state.get('sweep_id')}"
    )